# 03 · La historia de cada municipio y las variables para el modelo

> **Reto 2 del INE:** "¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene?"

**A qué responde este paso.** Para anticipar qué municipios votarán menos en la elección intermedia de 2027 necesitamos dos cosas:

1. **La historia limpia de cada municipio**, de 2009 a 2024, comparable entre años y con todas las decisiones de los pasos 1 y 2 aplicadas.
2. **Saber qué variables ayudan a anticipar** qué municipios quedan por debajo del resto, y cuáles no aportan nada.

Al final queda una tabla, `data/processed/panel_municipal.parquet`, con una fila por municipio y elección. Es la base del modelo (paso 4).

## Resumen de lo que encontramos

Cada hallazgo indica qué tan firme es: **verificado** (sale de los datos o de una fuente oficial), **indicio** (lo muestran los datos, pero con pocos casos o sin poder separar otras causas) o **hipótesis** (por comprobar).

1. **Verificado.** Entre 4.9 % y 6.5 % de la lista de cada elección vive en secciones que ya no existen en 2024, porque el INE las dividió. No se pierden: se asignan al municipio donde estaban. Solo en 13 municipios queda una duda real sobre una parte de su historia, y llevan la marca "territorio aproximado".
2. **Verificado.** Los 19 municipios creados después de 2009 recuperan su historia: 15 tienen el 100 % de su territorio en todas las elecciones.
3. **Verificado.** Las actas oficiales de las seis elecciones (2009 a 2024) confirman los datos: en la sección típica, nuestros "votó" son el 99 % de los votos del acta. Con ellas ya no queda ningún municipio "sin dato" por cuadernillos no capturados (2012) ni por el boicot de 2015.
4. **Verificado. Corrige el paso 2.** Hay 75 municipios donde, en alguna elección de 2009 a 2024, al menos el 10 % de la lista se quedó sin casilla (36 si solo se cuenta de 2018 a 2024). 16 repiten, y 8 en tres o más elecciones, como Santiago Yaveo (Oaxaca) y Cherán, Charapan y Tingambato (Michoacán). La fórmula de los conteos les **inflaba la participación**, 15 puntos en el caso típico.
5. **Indicio fuerte.** La historia del municipio es la mejor pista que tenemos (2018 anticipa dos tercios de las diferencias entre municipios en 2021). Lo que se elige localmente el mismo día también mueve la participación, y **corrige otra lectura del paso 2**: el gran efecto de 2015 no era solo del gobernador, sino de tener o no **alguna** elección local. La transición de 2024 a 2027 tiene la misma estructura que la de 2018 a 2021, que es la mejor referencia del pasado.
6. **Verificado.** La proporción de jóvenes y el crecimiento de la lista, por sí solos, casi no anticipan qué municipios votan menos. **Hipótesis:** la proporción de hombres aporta un poco, quizá como señal de migración.
7. **Verificado.** En el municipio típico, las personas de 20 a 29 años votan 9.9 puntos menos que el total (2021), y eso pasa en el 97.5 % de los municipios: el grupo a intervenir es el mismo en casi todo el país.

## Palabras que usamos

| Palabra | Qué significa |
|---|---|
| **Lista** | Lista nominal: las personas con credencial que podían votar en esa sección. |
| **Participación** | Votó / (votó + no votó). Las personas de las que **no sabemos** qué hicieron quedan fuera (decisión D1). |
| **pp** | Puntos porcentuales. Si un municipio pasa de 50 % a 55 %, subió 5 pp. |
| **Territorio de 2024** | Para comparar un municipio entre años, sumamos las secciones que en 2024 le pertenecen (decisión D6). |
| **Sección desaparecida** | Sección que existía en un año pero no en 2024, porque el INE la dividió en otras (reseccionamiento). |
| **Actas** | Resultados oficiales de la elección **casilla por casilla** (cómputos distritales del INE). Son una segunda fuente, independiente de los conteos censales. |
| **Sección comparable** | Sección donde todas sus casillas tienen acta, la lista del acta es igual a la nuestra y no hay "no sabemos". Ahí, "votó" y los votos del acta deberían coincidir. |
| **Desviación** | Participación del municipio menos la participación nacional de esa elección, en pp. Negativa = vota menos que el país. Es lo que el modelo va a anticipar. |
| **Origen A, B, C** | Por qué la fórmula no basta. A: parte de la lista sin casilla. B: el paquete no llegó. C: el cuadernillo no se capturó. |
| **R²** | Qué parte de las diferencias entre municipios en una elección se anticipa con otra (0 = nada, 1 = todo). |
| **Correlación** | Qué tanto se mueven juntas dos medidas (de −1 a 1; 0 = sin relación). |
| **Lo que la historia no anticipa** | La diferencia entre la desviación real de un municipio y la que se esperaba por su elección anterior. |

In [1]:
from pathlib import Path
import io
import zipfile
import hashlib
import urllib.request

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(14)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(180)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir())
datos = pl.scan_parquet(raiz / 'data/processed/ccpc/ccpc-*.parquet')
ANIOS = [2009, 2012, 2015, 2018, 2021, 2024]
TIPO = {2009: 'intermedia', 2012: 'presidencial', 2015: 'intermedia', 2018: 'presidencial', 2021: 'intermedia', 2024: 'presidencial'}

# Clave INEGI de cada municipio por año, con la corrección de Paracho y Parácuaro 2021 (notebook 01, sección 5)
equivalencias = pl.read_csv(raiz / 'data/reference/equivalencias_ine_inegi.csv', infer_schema=False).select(
    pl.col('anio').cast(pl.Int16).alias('AELEC'),
    pl.col('cve_ent_ine').cast(pl.Int8).alias('EDOCVE'),
    pl.col('cve_mun_ine').cast(pl.Int16).alias('MPIOCVE'),
    'cvegeo',
).with_columns(
    cvegeo=pl.when((pl.col('AELEC') == 2021) & (pl.col('EDOCVE') == 16) & (pl.col('MPIOCVE') == 65)).then(pl.lit('16064'))
    .when((pl.col('AELEC') == 2021) & (pl.col('EDOCVE') == 16) & (pl.col('MPIOCVE') == 66)).then(pl.lit('16065'))
    .otherwise(pl.col('cvegeo'))
)

### Actas oficiales de 2009 a 2024

Para cada elección usamos la que corresponde a la lista completa de cada sección:

- **presidencia** en 2012, 2018 y 2024;
- **diputaciones** en 2009, 2015 y 2021, que no tuvieron presidencial.

Todas son cómputos distritales oficiales del INE:

- **2018, 2021 y 2024:** vienen de los sitios de cómputos de cada elección. Si no están en la computadora, la siguiente celda las descarga a `data/raw/` (que no se sube al repositorio) y comprueba su huella digital (SHA-256).
- **2009, 2012 y 2015:** vienen del SICEEF del INE. Las procesa `scripts/externos/fuente_actas_2009_2015.py`, que también verifica las huellas. Coinciden con los acuerdos del Consejo General y con el cómputo publicado por el IFE, con diferencias de decenas de votos (`docs/fuentes-externas.md`).

In [2]:
FUENTES_ACTAS = {
    2018: ('https://computos2018.ine.mx/20180708_2130_CW.zip', '20180708_2130_CW_presidencia.zip', 'presidencia.csv', 'computos_2018/PRES_2018.csv'),
    2021: ('https://computos2021.ine.mx/20210611_1000_CW.zip', '20210611_1000_CW_diputaciones.zip', 'diputaciones.csv', 'computos_2021/DIP_2021.csv'),
    2024: ('https://computos2024.ine.mx/20240608_2030_COMPUTOS.zip', 'PRES.zip', 'PRES_2024.csv', 'computos_2024/PRES_2024.csv'),
}

def descargar_actas(url, zip_interno, csv_interno, destino):
    destino.parent.mkdir(parents=True, exist_ok=True)
    peticion = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    contenido = urllib.request.urlopen(peticion, timeout=300).read()
    with zipfile.ZipFile(io.BytesIO(contenido)) as z:
        interno = next(n for n in z.namelist() if n.endswith(zip_interno))
        with zipfile.ZipFile(io.BytesIO(z.read(interno))) as zi:
            nombre = next(n for n in zi.namelist() if n.endswith(csv_interno))
            destino.write_bytes(zi.read(nombre))

def leer_actas(ruta):
    lineas = ruta.read_text(encoding='latin-1').split('\n')
    inicio = next(i for i, linea in enumerate(lineas) if linea.startswith('CLAVE_CASILLA'))
    tabla = pl.read_csv(io.StringIO('\n'.join(lineas[inicio:])), separator='|', infer_schema=False, truncate_ragged_lines=True)
    # Los nombres de columna cambian un poco entre años
    entidad = 'ID_ENTIDAD' if 'ID_ENTIDAD' in tabla.columns else 'ID_ESTADO'
    lista = 'LISTA_NOMINAL' if 'LISTA_NOMINAL' in tabla.columns else 'LISTA_NOMINAL_CASILLA'
    limpia = lambda c: pl.col(c).str.replace_all(r"[=\"',]", '')
    return tabla.select(
        EDOCVE=limpia(entidad).cast(pl.Int8, strict=False),
        SECCION=limpia('SECCION').cast(pl.Int32, strict=False),
        tipo_casilla=pl.col('TIPO_CASILLA'),
        votos=limpia('TOTAL_VOTOS_CALCULADOS').cast(pl.Int64, strict=False),
        lista_casilla=limpia(lista).cast(pl.Int64, strict=False),
        estado=pl.when(pl.col('OBSERVACIONES').str.contains('(?i)casilla no instalada')).then(pl.lit('no instalada'))
        .when(pl.col('OBSERVACIONES').str.contains('(?i)paquete no (recibido|entregado)')).then(pl.lit('paquete no recibido'))
        .otherwise(pl.lit('computada')),
    # Básicas, contiguas y extraordinarias: las que corresponden a la lista de la sección
    ).filter(pl.col('tipo_casilla').is_in(['B', 'C', 'E']))

# Huella digital de los archivos con los que se hizo este análisis
HUELLAS = {
    2018: 'dda51b058b6d5ca15fbc26769381a9c4801f5a1cd11ff38c801a6db53bd347a2',
    2021: '2b7f6e3339e3a326336fee28b7665d92b3eece11db7199b9f7c4b5e3188cd1d5',
    2024: '860c1a539ac98b5def4dff3e82e87a19be3ebba7cb435b680ae79799942d1628',
}
actas_casilla = {}
for anio, (url, zip_interno, csv_interno, ruta) in FUENTES_ACTAS.items():
    archivo = raiz / 'data/raw' / ruta
    if not archivo.exists():
        descargar_actas(url, zip_interno, csv_interno, archivo)
    huella = hashlib.sha256(archivo.read_bytes()).hexdigest()
    assert huella == HUELLAS[anio], f'El archivo de {anio} no es el mismo con el que se hizo el análisis'
    actas_casilla[anio] = leer_actas(archivo).with_columns(extraordinaria=pl.lit(False))
    print(anio, archivo.name, 'SHA-256 verificada')

# 2009, 2012 y 2015: actas oficiales del SICEEF del INE, ya procesadas por scripts/externos/fuente_actas_2009_2015.py
for anio in [2009, 2012, 2015]:
    archivo = raiz / f'data/processed/externos/actas_{anio}.parquet'
    if not archivo.exists():
        print(f'{anio}: no están las actas; correr scripts/externos/fuente_actas_2009_2015.py')
        continue
    actas_casilla[anio] = pl.read_parquet(archivo).filter(pl.col('tipo_casilla').is_in(['B', 'C', 'E'])).select(
        'EDOCVE', 'SECCION', 'tipo_casilla',
        votos=pl.col('total_votos'),
        lista_casilla=pl.col('lista_casilla'),
        # Recontadas por el Tribunal: traen votos aunque digan "paquete no recibido". "Sin acta" y "otros" se tratan como paquete no recibido
        estado=pl.when((pl.col('estado') == 'paquete no recibido') & (pl.col('total_votos') > 0)).then(pl.lit('computada'))
        .when(pl.col('estado').is_in(['sin acta', 'otros'])).then(pl.lit('paquete no recibido'))
        .otherwise(pl.col('estado')),
        # Aguascalientes 01 (2015): el archivo trae la elección extraordinaria de diciembre, no la de junio
        extraordinaria=pl.col('eleccion_extraordinaria').fill_null(False),
    )
    print(anio, archivo.name, 'actas cargadas')
actas_casilla = dict(sorted(actas_casilla.items()))
ANIOS_ACTAS = list(actas_casilla)

pl.DataFrame([
    {'Elección': anio, 'Casillas': t.height,
     'Computadas': t.filter(pl.col('estado') == 'computada').height,
     'No instaladas': t.filter(pl.col('estado') == 'no instalada').height,
     'Paquete no recibido': t.filter(pl.col('estado') == 'paquete no recibido').height,
     'Casillas sin lista en el acta': t['lista_casilla'].null_count()}
    for anio, t in actas_casilla.items()
])

2018 PRES_2018.csv SHA-256 verificada
2021 DIP_2021.csv SHA-256 verificada
2024 PRES_2024.csv SHA-256 verificada
2009 actas_2009.parquet actas cargadas
2012 actas_2012.parquet actas cargadas


2015 actas_2015.parquet actas cargadas


Elección,Casillas,Computadas,No instaladas,Paquete no recibido,Casillas sin lista en el acta
i64,i64,i64,i64,i64,i64
2009,138403,138357,41,5,0
2012,142230,142215,2,13,0
2015,147940,147322,225,393,0
2018,155757,155555,11,191,202
2021,161697,161406,32,259,0
2024,169003,168859,23,121,0


**Qué vemos:** en 2018, **las casillas no instaladas y las de paquete no recibido no traen su lista nominal** (202 casillas). En las demás elecciones sí la traen.

**Qué hacemos:**

- **2018:** en esas secciones, la parte de la lista que ninguna casilla cubrió se reparte entre sus casillas sin lista, según cuántas son "no instalada" y cuántas "paquete no recibido". La celda siguiente lo comprueba en tres municipios de Chiapas. Una primera versión que no hacía este reparto los clasificaba por error como "sin casilla".
- **2012:** las casillas que dicen "paquete no recibido" pero traen votos (el Tribunal Electoral las recontó) se cuentan como computadas.
- **2009 y 2012:** las casillas "sin acta" (3) y "otros" (1) se tratan como paquete no recibido.
- **2015:** el distrito 01 de Aguascalientes trae la elección extraordinaria de diciembre, no la de junio. Sus secciones no se comparan contra los conteos (D10) ni se usan para medir participación.

**Qué casillas usamos:** básicas, contiguas y extraordinarias, que son las que reciben a la lista de cada sección. Quedan fuera las especiales y las mesas de voto anticipado, de prisión preventiva y del extranjero, que no corresponden a la lista de una sección.

In [3]:
# Comprobación: casillas de 2018 en Chanal, Chicoasén y San Andrés Duraznal (claves INEGI 07024, 07029 y 07118)
comprobar = equivalencias.filter((pl.col('AELEC') == 2018) & pl.col('cvegeo').is_in(['07024', '07029', '07118']))
secciones_comprobar = datos.filter(pl.col('AELEC') == 2018).select('EDOCVE', 'MPIOCVE', 'MPIONOM', 'SECCION', 'LN').unique(['EDOCVE', 'SECCION']).collect().join(comprobar, on=['EDOCVE', 'MPIOCVE'])
(
    actas_casilla[2018].join(secciones_comprobar.select('EDOCVE', 'SECCION', 'MPIONOM'), on=['EDOCVE', 'SECCION'])
    .group_by(pl.col('MPIONOM').alias('Municipio'), pl.col('estado').alias('Estado de la casilla')).len().sort('Municipio', 'Estado de la casilla')
    .rename({'len': 'Casillas'})
)

Municipio,Estado de la casilla,Casillas
str,str,u32
"""Chanal""","""computada""",7
"""Chanal""","""paquete no recibido""",8
"""Chicoasén""","""computada""",1
"""Chicoasén""","""paquete no recibido""",6
"""San Andrés Duraznal""","""computada""",1
"""San Andrés Duraznal""","""paquete no recibido""",5


## 1 · La historia de cada municipio con su territorio de 2024 (decisión D6)

**El problema.** Los municipios cambian: se crean nuevos (San Quintín se separó de Ensenada) y el INE divide las secciones que crecen mucho. Si comparamos "Ensenada 2009" con "Ensenada 2024", no es el mismo territorio.

**La regla.** Cada sección de cada año se asigna a un municipio de 2024:

1. **Si la sección sigue existiendo en 2024** (mismo estado y mismo número), va al municipio al que pertenece en 2024.
2. **Si ya no existe** (el INE la dividió), va al municipio al que pertenecía en su año.

¿Por qué funciona el paso 2? Porque en los datos, de las secciones que existen en un año y también en 2024, **las únicas que cambian de municipio son las de los municipios que cedieron territorio** (lo mostramos abajo). Así que la gente de una sección dividida sigue en el mismo municipio, salvo en esos casos.

Primero medimos cuánta gente vive en secciones que ya no existen en 2024.

In [4]:
joven = pl.col('EDAD').is_between(20, 29)  # grupo destacado en D8
hombre = pl.col('SEXO') == 0
secciones = (
    datos.group_by('AELEC', 'EDOCVE', 'MPIOCVE', 'SECCION')
    .agg(
        pl.col('TIPOSEC').first(),
        pl.col('LN', 'SV', 'NV', 'NS').sum(),
        LN_20_29=pl.col('LN').filter(joven).sum(),
        SV_20_29=pl.col('SV').filter(joven).sum(),
        NV_20_29=pl.col('NV').filter(joven).sum(),
        LN_h_20_29=pl.col('LN').filter(joven & hombre).sum(),
        SV_h_20_29=pl.col('SV').filter(joven & hombre).sum(),
        NV_h_20_29=pl.col('NV').filter(joven & hombre).sum(),
        LN_65_mas=pl.col('LN').filter(pl.col('EDAD') >= 65).sum(),
        LN_hombres=pl.col('LN').filter(hombre).sum(),
    )
    .collect()
    .join(equivalencias, on=['AELEC', 'EDOCVE', 'MPIOCVE'], how='left', validate='m:1')
)
assert secciones['cvegeo'].null_count() == 0, 'Hay secciones sin clave INEGI'

territorio_2024 = secciones.filter(pl.col('AELEC') == 2024).select('EDOCVE', 'SECCION', pl.col('cvegeo').alias('cvegeo_2024'))
secciones = secciones.join(territorio_2024, on=['EDOCVE', 'SECCION'], how='left').with_columns(
    seccion_existe_en_2024=pl.col('cvegeo_2024').is_not_null(),
    municipio_2024=pl.coalesce('cvegeo_2024', 'cvegeo'),
)

secciones.group_by('AELEC').agg(
    pl.len().alias('Secciones'),
    (~pl.col('seccion_existe_en_2024')).sum().alias('Secciones que ya no existen en 2024'),
    (pl.col('LN').filter(~pl.col('seccion_existe_en_2024')).sum() / pl.col('LN').sum() * 100).round(1).alias('% de la lista en ellas'),
).sort('AELEC').rename({'AELEC': 'Elección'})

Elección,Secciones,Secciones que ya no existen en 2024,% de la lista en ellas
i16,u32,u32,f64
2009,64934,1882,6.2
2012,65599,1748,6.3
2015,68362,1585,5.8
2018,68408,1598,6.5
2021,68812,836,4.9
2024,70751,0,0.0


**Qué vemos:** entre el 4.9 % y el 6.5 % de la lista de cada elección vive en secciones que ya no existen en 2024. Son más de 4.5 millones de personas por elección. **Si solo sumáramos las secciones de 2024, las perderíamos**; por eso existe el paso 2 de la regla.

### ¿Dónde puede fallar la regla?

Solo en los municipios que **cedieron territorio** a otro y **además** tienen secciones desaparecidas: no sabemos si esa gente vivía en la parte que se quedó o en la que se fue.

In [5]:
# Nombres oficiales del INEGI (los del archivo del INE traen algunos errores de captura)
nombres_estado = pl.read_csv(raiz / 'data/reference/elecciones_locales_concurrentes.csv', infer_schema=False).filter(pl.col('anio') == '2024').select(pl.col('cve_ent').alias('CVE_ENT'), pl.col('entidad').alias('estado'))
nombres_2024 = (
    equivalencias.filter(pl.col('AELEC') == 2024).select('cvegeo')
    .join(pl.read_csv(raiz / 'data/reference/municipios_inegi.csv', infer_schema=False), left_on='cvegeo', right_on='CVEGEO', how='left')
    .join(nombres_estado, on='CVE_ENT', how='left')
    .select('cvegeo', 'estado', pl.col('NOMGEO').alias('municipio'))
)
assert nombres_2024['municipio'].null_count() == 0 and nombres_2024['estado'].null_count() == 0
cedio = secciones.filter(pl.col('seccion_existe_en_2024') & (pl.col('cvegeo_2024') != pl.col('cvegeo')))
pares = (
    cedio.group_by(pl.col('cvegeo').alias('cedio'), pl.col('cvegeo_2024').alias('recibio')).agg(pl.col('AELEC').max().alias('ultimo_anio_juntos'))
    .join(nombres_2024.select(pl.col('cvegeo').alias('cedio'), pl.col('municipio').alias('Municipio que cedió territorio'), 'estado'), on='cedio', how='left')
    .join(nombres_2024.select(pl.col('cvegeo').alias('recibio'), pl.col('municipio').alias('Municipio que lo recibió')), on='recibio', how='left')
)
print(f'Municipios que cedieron territorio: {pares["cedio"].n_unique()} · pares: {pares.height}')
pares.select('estado', 'Municipio que cedió territorio', 'Municipio que lo recibió', pl.col('ultimo_anio_juntos').alias('Última elección en que su territorio estaba junto')).sort('estado', 'Municipio que cedió territorio')

Municipios que cedieron territorio: 25 · pares: 27


estado,Municipio que cedió territorio,Municipio que lo recibió,Última elección en que su territorio estaba junto
str,str,str,i16
"""Baja California""","""Ensenada""","""San Quintín""",2021
"""Baja California""","""Ensenada""","""San Felipe""",2021
"""Baja California""","""Mexicali""","""San Felipe""",2021
"""Campeche""","""Calkiní""","""Dzitbalché""",2018
"""Campeche""","""Champotón""","""Seybaplaya""",2018
"""Chiapas""","""Acala""","""Emiliano Zapata""",2018
"""Chiapas""","""Pueblo Nuevo Solistahuacán""","""Rincón Chamula San Pedro""",2018
"""Chiapas""","""Siltepec""","""Honduras de la Sierra""",2018
"""Chiapas""","""Siltepec""","""Capitán Luis Ángel Vidal""",2018


**Qué vemos:** casi todos los pares son los 19 municipios creados después de 2009, en Baja California, Campeche, Chiapas, Guerrero, Morelos y Sinaloa. El resto son ajustes de límites entre municipios que ya existían: Apodaca y Guadalupe (Nuevo León), Álvaro Obregón y Cuajimalpa (Ciudad de México), y La Magdalena Tlaltelulco y Chiautempan (Tlaxcala).

**¿Cuándo queda la duda de verdad?** Medimos qué parte de la lista de 2024 de cada municipio vive en secciones que ya existían en cada elección (su **cobertura**). Si el municipio que recibió territorio tiene el 100 % de cobertura en un año, todas sus secciones ya existían: las secciones desaparecidas del que cedió no pueden ser suyas, y la historia de los dos es exacta. **La duda solo existe cuando el que recibió no tiene todo su territorio ese año.** En ese caso marcamos **a los dos** con "territorio aproximado" (decisión D13 ajustada).

### ¿Recuperan su historia los municipios nuevos?

Para cada municipio creado después de 2009 (los que no existen en la tabla de equivalencias de 2009), esta es su cobertura en cada elección.

In [6]:
cobertura = []
for anio in ANIOS[:-1]:
    existian = secciones.filter(pl.col('AELEC') == anio).select('EDOCVE', 'SECCION').with_columns(existia=pl.lit(True))
    cobertura.append(
        secciones.filter(pl.col('AELEC') == 2024).select('EDOCVE', 'SECCION', 'cvegeo', 'LN')
        .join(existian, on=['EDOCVE', 'SECCION'], how='left')
        .group_by('cvegeo').agg((pl.col('LN').filter(pl.col('existia')).sum() / pl.col('LN').sum() * 100).round(1).alias(str(anio)))
    )
cobertura = cobertura[0].join(cobertura[1], on='cvegeo').join(cobertura[2], on='cvegeo').join(cobertura[3], on='cvegeo').join(cobertura[4], on='cvegeo')

# D13 ajustada: duda solo si quien recibió no tiene todo su territorio ese año (y quien cedió tiene secciones desaparecidas)
cobertura_larga = cobertura.unpivot(index='cvegeo', variable_name='AELEC', value_name='cobertura').with_columns(pl.col('AELEC').cast(pl.Int16))
con_desaparecidas = secciones.filter(~pl.col('seccion_existe_en_2024')).select(pl.col('cvegeo').alias('cedio'), 'AELEC').unique()
dudosos = (
    pares.select('cedio', 'recibio', 'ultimo_anio_juntos')
    .join(pl.DataFrame({'AELEC': ANIOS[:-1]}, schema={'AELEC': pl.Int16}), how='cross')
    .filter(pl.col('AELEC') <= pl.col('ultimo_anio_juntos'))
    .join(cobertura_larga.rename({'cvegeo': 'recibio'}), on=['recibio', 'AELEC'])
    .join(con_desaparecidas, on=['cedio', 'AELEC'], how='semi')
    .filter(pl.col('cobertura') < 99)
)
territorio_aproximado = pl.concat([dudosos.select(pl.col('cedio').alias('cvegeo'), 'AELEC'), dudosos.select(pl.col('recibio').alias('cvegeo'), 'AELEC')]).unique()
print(f'Municipio-elección con "territorio aproximado": {territorio_aproximado.height} · municipios: {territorio_aproximado["cvegeo"].n_unique()}')
print(dudosos.join(nombres_2024.select(pl.col('cvegeo').alias('cedio'), pl.col('municipio').alias('cedió')), on='cedio').join(nombres_2024.select(pl.col('cvegeo').alias('recibio'), pl.col('municipio').alias('recibió')), on='recibio')
      .group_by('cedió', 'recibió').agg(pl.col('AELEC').sort().alias('elecciones'), pl.col('cobertura').min().alias('cobertura mínima de quien recibió (%)')).sort('cedió'))
nuevos = sorted(set(equivalencias.filter(pl.col('AELEC') == 2024)['cvegeo']) - set(equivalencias.filter(pl.col('AELEC') == 2009)['cvegeo']))
print(f'Municipios creados después de 2009: {len(nuevos)}')
(
    cobertura.filter(pl.col('cvegeo').is_in(nuevos))
    .join(nombres_2024, on='cvegeo').sort('cvegeo')
    .select('estado', pl.col('municipio').alias('Municipio nuevo'), *[str(a) for a in ANIOS[:-1]])
)

Municipio-elección con "territorio aproximado": 27 · municipios: 13
shape: (9, 4)
┌─────────────────┬────────────────┬──────────────────────┬───────────────────────────────────────┐
│ cedió           ┆ recibió        ┆ elecciones           ┆ cobertura mínima de quien recibió (%) │
│ ---             ┆ ---            ┆ ---                  ┆ ---                                   │
│ str             ┆ str            ┆ list[i16]            ┆ f64                                   │
╞═════════════════╪════════════════╪══════════════════════╪═══════════════════════════════════════╡
│ Ahome           ┆ Juan José Ríos ┆ [2012]               ┆ 96.3                                  │
│ Apodaca         ┆ Guadalupe      ┆ [2009, 2012, … 2021] ┆ 86.8                                  │
│ Culiacán        ┆ Eldorado       ┆ [2012]               ┆ 95.0                                  │
│ El Fuerte       ┆ Juan José Ríos ┆ [2012]               ┆ 96.3                                  │
│ Guadalupe       

estado,Municipio nuevo,2009,2012,2015,2018,2021
str,str,f64,f64,f64,f64,f64
"""Baja California""","""San Quintín""",100.0,100.0,100.0,100.0,100.0
"""Baja California""","""San Felipe""",100.0,100.0,100.0,100.0,100.0
"""Campeche""","""Seybaplaya""",100.0,100.0,100.0,100.0,100.0
"""Campeche""","""Dzitbalché""",100.0,100.0,100.0,100.0,100.0
"""Chiapas""","""Capitán Luis Ángel Vidal""",100.0,100.0,100.0,100.0,100.0
"""Chiapas""","""Rincón Chamula San Pedro""",100.0,100.0,100.0,100.0,100.0
"""Chiapas""","""El Parral""",100.0,100.0,100.0,100.0,100.0
"""Chiapas""","""Emiliano Zapata""",100.0,100.0,100.0,100.0,100.0
"""Chiapas""","""Mezcalapa""",100.0,100.0,100.0,100.0,100.0


**Qué vemos:**

- **15 de los 19 municipios nuevos tienen el 100 % de su territorio en todas las elecciones.** Sus secciones ya existían, solo que dentro de otro municipio.
- **Las Vigas, Eldorado y Juan José Ríos** bajan a 95–96 % solo en 2012, por los cuadernillos que el INE no capturó ese año, no por el territorio.
- **Xoxocotla** tiene el 82 % en todos los años, porque una de sus secciones se dividió. Su historia y la de Puente de Ixtla quedan con la marca "territorio aproximado".

**Interpretación:** con la regla de D6, los municipios nuevos ya no aparecen "sin dato" en sus primeros años: tienen historia desde 2009 y el modelo los puede evaluar igual que a los demás.

## 2 · Las actas confirman los datos de 2009 a 2024 (decisiones D9 y D10)

En el paso 2 cruzamos los datos con las actas de 2024. Ahora lo hacemos con las seis elecciones, sección por sección:

- **Secciones comparables:** aquellas donde todas sus casillas tienen acta computada, la lista del acta es igual a la nuestra y no hay "no sabemos". Ahí, "votó" debería ser casi igual a los votos del acta.
- **D10:** si nuestros datos marcan más de 10 % (y al menos 50) **más** "votó" que votos el acta, la sección pasa a "sin dato" (sus personas cuentan como "no sabemos").

In [7]:
partes = []
for anio, tabla in actas_casilla.items():
    partes.append(tabla.group_by('EDOCVE', 'SECCION').agg(
        actas_lista_computada=pl.col('lista_casilla').filter(pl.col('estado') == 'computada').sum(),
        actas_lista_paquete=pl.col('lista_casilla').filter(pl.col('estado') == 'paquete no recibido').sum(),
        actas_votos=pl.col('votos').filter(pl.col('estado') == 'computada').sum(),
        actas_casillas=pl.len(),
        paquete_sin_lista=(pl.col('lista_casilla').is_null() & (pl.col('estado') == 'paquete no recibido')).sum(),
        no_instalada_sin_lista=(pl.col('lista_casilla').is_null() & (pl.col('estado') == 'no instalada')).sum(),
        actas_extraordinaria=pl.col('extraordinaria').any(),
    ).with_columns(AELEC=pl.lit(anio, pl.Int16)))
actas_seccion = pl.concat(partes)

columnas_actas = ['actas_lista_computada', 'actas_lista_paquete', 'actas_votos', 'actas_casillas', 'paquete_sin_lista', 'no_instalada_sin_lista']
secciones = (
    secciones.join(actas_seccion, on=['AELEC', 'EDOCVE', 'SECCION'], how='left')
    .with_columns(hay_actas=pl.col('AELEC').is_in(ANIOS_ACTAS))
    .with_columns(pl.when(pl.col('hay_actas')).then(pl.col(c).fill_null(0)).otherwise(pl.col(c)).alias(c) for c in columnas_actas)
    .with_columns(
        no_cubierta=(pl.col('LN') - pl.col('actas_lista_computada') - pl.col('actas_lista_paquete')).clip(lower_bound=0),
        casillas_sin_lista=pl.col('paquete_sin_lista') + pl.col('no_instalada_sin_lista'),
    )
    .with_columns(
        # 2018: la lista no cubierta se reparte entre las casillas sin lista según su tipo
        actas_lista_paquete=pl.min_horizontal(
            pl.col('actas_lista_paquete')
            + pl.when(pl.col('casillas_sin_lista') > 0).then(pl.col('no_cubierta') * pl.col('paquete_sin_lista') / pl.col('casillas_sin_lista')).otherwise(0).round(0).cast(pl.Int64),
            pl.col('LN'),
        ),
    )
    .with_columns(
        # Lista sin casilla: la que no cubrió ninguna casilla que funcionó (computada o con paquete perdido)
        actas_sin_casilla=(pl.col('LN') - pl.col('actas_lista_computada') - pl.col('actas_lista_paquete')).clip(lower_bound=0),
        comparable=pl.col('hay_actas') & ~pl.col('actas_extraordinaria').fill_null(False) & (pl.col('LN') == pl.col('actas_lista_computada')) & (pl.col('NS') == 0) & (pl.col('actas_lista_paquete') == 0),
    )
    .with_columns(
        votaron_de_mas=pl.col('comparable') & (pl.col('SV') > pl.col('actas_votos') * 1.10) & (pl.col('SV') - pl.col('actas_votos') >= 50),
        votaron_de_menos=pl.col('comparable') & (pl.col('SV') < pl.col('actas_votos') * 0.90) & (pl.col('actas_votos') - pl.col('SV') >= 50),
    )
    .with_columns(no_cuadra_con_acta=pl.col('votaron_de_mas') | pl.col('votaron_de_menos'))
)

(
    secciones.filter(pl.col('hay_actas')).group_by('AELEC').agg(
        pl.col('comparable').sum().alias('Secciones comparables'),
        (pl.col('SV') / pl.col('actas_votos')).filter(pl.col('comparable')).median().round(3).alias('Votó / votos del acta (mediana)'),
        pl.col('votaron_de_mas').sum().alias('Más "votó" que votos del acta (D10)'),
        pl.col('votaron_de_menos').sum().alias('Menos "votó" que votos del acta'),
        (pl.col('LN').filter(pl.col('no_cuadra_con_acta')).sum() / pl.col('LN').sum() * 100).round(2).alias('% de la lista en secciones que no cuadran'),
    ).sort('AELEC').rename({'AELEC': 'Elección'})
)

Elección,Secciones comparables,Votó / votos del acta (mediana),"Más ""votó"" que votos del acta (D10)","Menos ""votó"" que votos del acta",% de la lista en secciones que no cuadran
i16,u32,f64,u32,u32,f64
2009,62080,0.989,51,224,0.46
2012,57645,0.992,145,189,0.47
2015,59341,0.988,252,175,0.59
2018,58019,0.99,819,354,1.58
2021,61367,0.99,317,225,0.71
2024,61139,0.992,359,285,0.83


**Qué vemos:**

- **En las seis elecciones, la sección típica cuadra entre el 98.8 % y el 99.2 %** con su acta. Los conteos censales y las actas son fuentes independientes, así que esto confirma la calidad de los datos.
- **2024 reproduce el paso 2:** 61,139 secciones comparables y 359 donde nuestros datos marcan de más.
- **También hay secciones donde nuestros datos marcan de menos:** entre 175 y 354 por elección, con menos "votó" que votos en el acta por más de 10 % y al menos 50 votos. En el paso 2 solo revisamos el otro sentido. Para el reto este error importa igual o más, porque **fabrica abstención que no existió**.
- **En total, las secciones que no cuadran tienen entre 0.5 % y 0.8 % de la lista**, salvo en 2018 (1.6 %).

**Qué hacemos (D10 ajustada, sección 8):** las secciones que no cuadran, en cualquiera de los dos sentidos, pasan a "no sabemos". No buscamos su causa una por una.

In [8]:
d10 = pl.col('no_cuadra_con_acta').fill_null(False)
secciones = secciones.with_columns(
    NS=pl.when(d10).then(pl.col('NS') + pl.col('SV') + pl.col('NV')).otherwise(pl.col('NS')),
    SV=pl.when(d10).then(0).otherwise(pl.col('SV')),
    NV=pl.when(d10).then(0).otherwise(pl.col('NV')),
    *[pl.when(d10).then(0).otherwise(pl.col(c)).alias(c) for c in ['SV_20_29', 'NV_20_29', 'SV_h_20_29', 'NV_h_20_29']],
)

sumas = ['LN', 'SV', 'NV', 'NS', 'LN_20_29', 'SV_20_29', 'NV_20_29', 'LN_h_20_29', 'SV_h_20_29', 'NV_h_20_29', 'LN_65_mas', 'LN_hombres',
         'actas_lista_computada', 'actas_lista_paquete', 'actas_votos', 'actas_sin_casilla']
panel = secciones.group_by('AELEC', pl.col('municipio_2024').alias('cvegeo')).agg(
    pl.col(sumas).sum(),
    pl.col('hay_actas').first(),
    LN_rural=pl.col('LN').filter(pl.col('TIPOSEC') == 'R').sum(),
    secciones_sin_casilla=(pl.col('hay_actas') & (pl.col('actas_casillas') == 0)).sum(),
    LN_secciones_desaparecidas=pl.col('LN').filter(~pl.col('seccion_existe_en_2024')).sum(),
    secciones_que_no_cuadran=d10.sum(),
)
malla = nombres_2024.select('cvegeo').join(pl.DataFrame({'AELEC': ANIOS}, schema={'AELEC': pl.Int16}), how='cross')
panel = (
    malla.join(panel, on=['AELEC', 'cvegeo'], how='left')
    .join(nombres_2024, on='cvegeo')
    .join(territorio_aproximado.with_columns(territorio_aproximado=pl.lit(True)), on=['cvegeo', 'AELEC'], how='left')
    .with_columns(pl.col('territorio_aproximado').fill_null(False), pl.col('hay_actas').fill_null(pl.col('AELEC').is_in(ANIOS_ACTAS)))
    # Años sin actas: esas columnas quedan vacías, no en 0
    .with_columns(pl.when(pl.col('hay_actas')).then(pl.col(c)).alias(c) for c in ['actas_lista_computada', 'actas_lista_paquete', 'actas_votos'])
    # Lista sin casilla medida en el municipio completo: las secciones pequeñas que votan en la casilla de una vecina no cuentan como "sin casilla"
    .with_columns(actas_sin_casilla=(pl.col('LN') - pl.col('actas_lista_computada') - pl.col('actas_lista_paquete')).clip(lower_bound=0))
    .sort('cvegeo', 'AELEC')
)
assert panel.filter(~pl.col('hay_actas'))['actas_sin_casilla'].null_count() == panel.filter(~pl.col('hay_actas')).height
print(f'Municipios de 2024: {nombres_2024.height:,} · filas municipio-elección: {panel.height:,} · sin ningún registro: {panel["LN"].null_count()}')

Municipios de 2024: 2,477 · filas municipio-elección: 14,862 · sin ningún registro: 3


## 3 · Cuándo un municipio queda "sin dato" y de dónde sale su participación (D2, D3 y D16)

Aplicamos las reglas decididas:

- **D2. "Sin dato"** si:
  1. no hay registros;
  2. no sabemos lo que hizo más de la mitad de la lista y las actas no lo resuelven;
  3. la lista es menor al 80 % de la de sus elecciones vecinas y no hay actas;
  4. casi nadie votó (menos de 2 %), sin "no sabemos" y con participación normal en sus otras elecciones.
- **D3 ajustada.** Con las actas (todas las elecciones de 2009 a 2024) se identifica el origen del problema:
  - **A, parte de la lista sin casilla:** al menos 10 % de la lista del municipio no quedó cubierta por ninguna casilla que funcionara. **Se revisa siempre**, aunque el "no sabemos" sea menor a la mitad (ver abajo por qué).
  - **B, paquete no recibido:** más de la mitad en "no sabemos" y 25 % o más de la lista con paquete perdido.
  - **C, cuadernillo no capturado:** más de la mitad en "no sabemos", o lista incompleta (D16), por cualquier otro motivo.

  La participación sale de las actas: **votos / (lista − lista con paquete perdido)**. Así, quienes no tuvieron casilla cuentan como personas que no votaron, porque no pudieron hacerlo. Si a menos de la mitad de la lista le llegó su acta, el año queda "sin dato".
- **D16 (nueva, propuesta).** Cuando a los conteos les faltan secciones completas (lista incompleta, sobre todo en 2012, por los cuadernillos que el INE no capturó), la participación sale de **todas las actas del municipio: votos / lista de las casillas computadas**. Las secciones que no están en los conteos de ese año se ubican con el mismo número de sección en la elección más cercana.

**Por qué revisamos la lista sin casilla siempre.** En una primera versión solo la buscábamos cuando más de la mitad estaba en "no sabemos". Los revisores encontraron municipios como Tingambato o Charapan (Michoacán), donde un tercio de la lista se queda sin casilla en cada elección. Como el "no sabemos" no pasaba de la mitad, usaban la fórmula de los conteos, que deja fuera a quienes no tuvieron casilla y les **inflaba la participación** hasta en 30 puntos. Lo medimos **en el municipio completo**: las secciones pequeñas que votan en la casilla de una sección vecina no cuentan como "sin casilla". Medida así, en el 99 % de los municipio-elección la lista sin casilla es menor a 3 %, por lo que el umbral de 10 % está muy por encima del ruido.

**Dos ajustes a la regla 3** (D2 ajustada, sección 8):

- **Solo se aplica cuando hay elección antes y después** (2012 a 2021). En 2009 solo existe la elección siguiente, y la regla marcaba a municipios que de verdad crecieron rápido entre 2009 y 2012, como García y Juárez (Nuevo León) o Solidaridad (Quintana Roo).
- **Las elecciones vecinas se promedian con la media geométrica.** La media simple marcaba a Pesquería (Nuevo León), cuya lista se duplicó en cada elección (9,937, 13,539, 25,840 y 50,678): 2012 quedaba por debajo del 80 % del promedio de 2009 y 2015, aunque su crecimiento era real.

In [9]:
# D16: total de actas por municipio. Las secciones que faltan en los conteos de ese año (cuadernillos no capturados)
# se ubican con el mismo número de sección en la elección más cercana.
ubicacion = secciones.select('AELEC', 'EDOCVE', 'SECCION', 'municipio_2024').unique()
partes = []
for anio, tabla in actas_casilla.items():
    mapa = (ubicacion.with_columns(distancia=(pl.col('AELEC') - anio).abs()).sort('distancia', 'AELEC')
            .unique(['EDOCVE', 'SECCION'], keep='first').select('EDOCVE', 'SECCION', 'municipio_2024'))
    t = tabla.filter(~pl.col('extraordinaria')).join(mapa, on=['EDOCVE', 'SECCION'], how='left')
    assert t['municipio_2024'].null_count() == 0, f'{anio}: hay casillas sin municipio'
    partes.append(t.group_by(pl.col('municipio_2024').alias('cvegeo')).agg(
        actas_municipio_votos=pl.col('votos').filter(pl.col('estado') == 'computada').sum(),
        actas_municipio_lista=pl.col('lista_casilla').filter(pl.col('estado') == 'computada').sum(),
    ).with_columns(AELEC=pl.lit(anio, pl.Int16)))
panel = panel.join(pl.concat(partes), on=['cvegeo', 'AELEC'], how='left')

mucho_no_sabemos = pl.col('pct_no_sabemos') > 50
lista_incompleta = pl.col('AELEC').is_between(2012, 2021) & (pl.col('LN') < 0.8 * pl.col('lista_vecinas'))
panel = panel.with_columns(
    pct_no_sabemos=pl.col('NS') / pl.col('LN') * 100,
    participacion_conteo=pl.when(pl.col('SV') + pl.col('NV') > 0).then(pl.col('SV') / (pl.col('SV') + pl.col('NV')) * 100),
    pct_sin_casilla=pl.col('actas_sin_casilla') / pl.col('LN') * 100,
    pct_paquete=pl.col('actas_lista_paquete') / pl.col('LN') * 100,
    # Media geométrica de la elección anterior y la siguiente
    lista_vecinas=(pl.col('LN').shift(1).over('cvegeo', order_by='AELEC') * pl.col('LN').shift(-1).over('cvegeo', order_by='AELEC')).sqrt(),
).with_columns(
    origen=pl.when(pl.col('hay_actas') & (pl.col('pct_sin_casilla') >= 10)).then(pl.lit('A · parte de la lista sin casilla'))
    .when(pl.col('hay_actas') & mucho_no_sabemos & (pl.col('pct_paquete') >= 25)).then(pl.lit('B · paquete no recibido'))
    .when(pl.col('hay_actas') & (mucho_no_sabemos | lista_incompleta)).then(pl.lit('C · cuadernillo no capturado')),
    lista_con_acta=pl.col('LN') - pl.col('actas_lista_paquete'),
).with_columns(
    participacion_actas=pl.when(lista_incompleta & ~mucho_no_sabemos & (pl.col('actas_municipio_lista') > 0))
    .then(pl.col('actas_municipio_votos') / pl.col('actas_municipio_lista') * 100)
    .when(pl.col('lista_con_acta') >= 0.5 * pl.col('LN')).then(pl.col('actas_votos') / pl.col('lista_con_acta') * 100),
    participacion_otras_elecciones=pl.col('participacion_conteo').filter(pl.col('pct_no_sabemos') <= 50).median().over('cvegeo'),
)
motivo = (
    pl.when(pl.col('LN').is_null()).then(pl.lit('1 · sin registros'))
    .when(mucho_no_sabemos & pl.col('origen').is_null()).then(pl.lit('2 · más de la mitad en "no sabemos", sin actas'))
    .when(pl.col('origen').is_not_null() & pl.col('participacion_actas').is_null()).then(pl.lit('2 · paquete perdido para más de la mitad de la lista'))
    .when(lista_incompleta & pl.col('origen').is_null()).then(pl.lit('3 · lista incompleta, sin actas'))
    .when((pl.col('participacion_conteo') < 2) & (pl.col('pct_no_sabemos') < 1) & (pl.col('participacion_otras_elecciones') > 20)).then(pl.lit('4 · casi nadie votó (posible error de registro)'))
)
panel = panel.with_columns(motivo_sin_dato=motivo).with_columns(
    participacion=pl.when(pl.col('motivo_sin_dato').is_not_null()).then(None)
    .when(pl.col('origen').is_not_null()).then(pl.col('participacion_actas'))
    .otherwise(pl.col('participacion_conteo')),
    fuente_participacion=pl.when(pl.col('motivo_sin_dato').is_not_null()).then(pl.lit('sin dato'))
    .when(pl.col('origen').is_not_null()).then(pl.lit('actas'))
    .otherwise(pl.lit('conteos censales')),
)

(
    panel.filter(pl.col('motivo_sin_dato').is_not_null())
    .group_by('motivo_sin_dato', 'AELEC').len()
    .pivot(on='AELEC', index='motivo_sin_dato', values='len', sort_columns=True)
    .fill_null(0).sort('motivo_sin_dato').rename({'motivo_sin_dato': 'Motivo de "sin dato"'})
)

"Motivo de ""sin dato""",2012,2015,2018,2021,2024
str,u32,u32,u32,u32,u32
"""1 · sin registros""",3,0,0,0,0
"""2 · paquete perdido para más de la mitad de la lista""",0,14,8,7,2
"""4 · casi nadie votó (posible error de registro)""",4,1,0,0,0


**Qué vemos:**

- **Con las actas de las seis elecciones, ya no queda ningún municipio "sin dato" por lista incompleta ni por "no sabemos" sin actas.** Los 108 municipios de 2012 con lista incompleta (cuadernillos no capturados) y los 67 de 2015 con mucho "no sabemos" ahora tienen participación de las actas.
- **Quedan "sin dato" 39 municipio-elección:**
  - **31 donde el paquete se perdió para más de la mitad de la lista:** 14 en 2015 (12 en Oaxaca), 8 en 2018, 7 en 2021 y 2 en 2024.
  - **3 sin ningún registro en 2012:** San Andrés Duraznal (Chiapas), y San Francisco Nuxaño y San Juan Teita (Oaxaca).
  - **5 donde casi nadie votó, sin "no sabemos" (regla 4):** San Mateo Sindihui, San Miguel Piedras, San Pedro Teozacoalco y Yutanduchi de Guerrero (Oaxaca, 2012), y Santa Magdalena Jicotlán (Oaxaca, 2015). Lo más probable es un error de registro, pero **no lo hemos verificado**.
- **Un caso a vigilar:** Hueyapan (Morelos) en 2021. A sus conteos les faltan secciones (paso 2, reporte E). Según sus actas, votó el 31.4 %, contra 71.5 % en 2018 y 61.6 % en 2024. **No conocemos la causa.**

**Conciliación con el paso 1:** allá contamos 73 municipio-elección "sin ningún dato", que eran municipios con **todo** en "no sabemos". Aquí la fila "sin registros" cuenta otra cosa: municipios de 2024 sin **ninguna** fila en esa elección.

Ahora, los municipios cuya participación sale de las actas, por origen:

In [10]:
(
    panel.filter(pl.col('origen').is_not_null())
    .group_by('AELEC', 'origen').len()
    .pivot(on='origen', index='AELEC', values='len', sort_columns=True).fill_null(0).sort('AELEC')
    .rename({'AELEC': 'Elección'})
)

Elección,A · parte de la lista sin casilla,B · paquete no recibido,C · cuadernillo no capturado
i16,u32,u32,u32
2009,8,0,14
2012,2,0,111
2015,39,13,28
2018,11,11,52
2021,25,11,14
2024,18,5,27


In [11]:
(
    panel.filter(pl.col('origen').str.starts_with('A'))
    .select(
        pl.col('AELEC').alias('Elección'), 'estado', 'municipio',
        pl.col('pct_no_sabemos').round(1).alias('% "no sabemos"'),
        pl.col('pct_sin_casilla').round(1).alias('% de la lista sin casilla'),
        pl.col('participacion_conteo').round(1).alias('Participación con la fórmula (%)'),
        pl.col('participacion').round(1).alias('Participación según actas, sobre toda la lista (%)'),
    ).sort('estado', 'municipio', 'Elección')
)

Elección,estado,municipio,"% ""no sabemos""",% de la lista sin casilla,Participación con la fórmula (%),"Participación según actas, sobre toda la lista (%)"
i16,str,str,f64,f64,f64,f64
2024,"""Chiapas""","""Chicomuselo""",100.0,100.0,null,0.0
2021,"""Chiapas""","""Honduras de la Sierra""",100.0,100.0,null,0.0
2024,"""Chiapas""","""Honduras de la Sierra""",66.6,24.0,52.2,42.9
2021,"""Chiapas""","""Ixhuatán""",59.6,30.7,80.8,56.4
2021,"""Chiapas""","""Oxchuc""",100.0,100.0,null,0.0
2024,"""Chiapas""","""Pantelhó""",100.0,100.0,null,0.0
2015,"""Chiapas""","""Pueblo Nuevo Solistahuacán""",19.6,14.7,48.2,42.1
2021,"""Chiapas""","""Pueblo Nuevo Solistahuacán""",28.1,11.7,78.6,70.9
2021,"""Chiapas""","""Siltepec""",100.0,100.0,null,0.0


**Qué vemos:**

- **De 2009 a 2024 hay 103 casos municipio-elección en 75 municipios** con al menos 10 % de la lista sin casilla. Son 54 casos en 36 municipios si solo se cuenta de 2018 a 2024.
- **En 32 casos nadie pudo votar** (100 % de la lista sin casilla): 3 en 2009, 1 en 2012, 18 en 2015, 6 en 2021 y 4 en 2024.
- **2015 es el año con más casos (39), y 34 son de Oaxaca.** Coincide con el boicot magisterial documentado en el paso 2. Las cifras coinciden con lo que el reporte A había medido por otro camino:

  | Municipio | Lista sin casilla, ahora | Reporte A |
  | --- | --- | --- |
  | Huautla de Jiménez | 93.5 % | 94 % |
  | San José Tenango | 84.1 % | 84 % |
  | Santa María Chilchotla | 55.9 % | 56 % |
- **La falta de casillas se repite:** 16 municipios la tuvieron en dos o más elecciones, y 8 en tres o más:
  - Santiago Yaveo (Oaxaca), Cherán, Charapan y Tingambato (Michoacán), en cuatro;
  - San Mateo del Mar (Oaxaca), Tila (Chiapas), y Nahuatzen y Quiroga (Michoacán), en tres.
- **La fórmula de los conteos censales sobrestima su participación:** 15 pp en el caso típico. Cherán marca 47.1 % con la fórmula en 2024, pero su participación sobre toda la lista fue de 7.1 %.

**Qué sabemos de sus causas:**

- **Documentadas en el paso 2 (reporte A):**
  - Cherán (desde 2015), Bolaños y Nahuatzen;
  - el boicot magisterial de 2015 en Oaxaca y en Tixtla, Guerrero;
  - San Mateo del Mar en 2012 y 2024;
  - Oxchuc, Venustiano Carranza, Honduras de la Sierra y Siltepec en 2021;
  - Chicomuselo y Pantelhó en 2024.
- **Corrección al reporte A:** ponía a Santa María Xadani (2021) entre las quemas de paquetes, a partir de prensa. **Las actas oficiales registran sus 11 casillas como "Casilla no instalada"**, así que prevalece el acta.
- **Sin causa documentada todavía:** el resto, incluidos los recurrentes Santiago Yaveo, Tila, Charapan, Quiroga, Tingambato, Mezquitic, Chilchota y Ziracuaretiro. Quedan como búsquedas pendientes (reporte H). **No suponemos la causa.**

Ahora los de origen B (paquete no recibido):

In [12]:
(
    panel.filter(pl.col('origen').str.starts_with('B'))
    .select(
        pl.col('AELEC').alias('Elección'), 'estado', 'municipio',
        pl.col('pct_paquete').round(1).alias('% de la lista con paquete no recibido'),
        pl.col('participacion').round(1).alias('Participación según actas (%)'),
        pl.col('motivo_sin_dato').alias('Sin dato porque'),
    ).sort('Elección', 'estado', 'municipio')
)

Elección,estado,municipio,% de la lista con paquete no recibido,Participación según actas (%),Sin dato porque
i16,str,str,f64,f64,str
2015,"""Oaxaca""","""Magdalena Tequisistlán""",82.5,null,"""2 · paquete perdido para más de la mitad de la lista"""
2015,"""Oaxaca""","""Putla Villa de Guerrero""",51.1,null,"""2 · paquete perdido para más de la mitad de la lista"""
2015,"""Oaxaca""","""San Andrés Huaxpaltepec""",66.4,null,"""2 · paquete perdido para más de la mitad de la lista"""
2015,"""Oaxaca""","""San Juan Cacahuatepec""",70.4,null,"""2 · paquete perdido para más de la mitad de la lista"""
2015,"""Oaxaca""","""San Lorenzo""",51.6,null,"""2 · paquete perdido para más de la mitad de la lista"""
2015,"""Oaxaca""","""San Miguel Panixtlahuaca""",100.0,null,"""2 · paquete perdido para más de la mitad de la lista"""
2015,"""Oaxaca""","""San Miguel Suchixtepec""",100.0,null,"""2 · paquete perdido para más de la mitad de la lista"""
2015,"""Oaxaca""","""San Pedro Tapanatepec""",58.2,null,"""2 · paquete perdido para más de la mitad de la lista"""
2015,"""Oaxaca""","""Santa María Jalapa del Marqués""",81.6,null,"""2 · paquete perdido para más de la mitad de la lista"""


**Qué vemos:**

- **Los de origen B se concentran** en Oaxaca en 2015 (12 de 13), en Chiapas en 2018 (6 de 11) y en Oaxaca, Puebla y Yucatán en 2021 (7 de 11).
- **Varios coinciden con las quemas o pérdidas de paquetes del reporte A** del paso 2: Xocchel y Sanahcat (2018); Laollaga, Huehuetlán, Chiconcuautla, Uayma y Yaxkukul (2021); Rincón Chamula (2018 y 2024); y la zona del boicot de 2015.
- **Los demás no tienen causa documentada:**
  - 2018: Chanal, Chicoasén, Montecristo de Guerrero, Tapilula, San Andrés Duraznal, Lagunillas, Ahuazotepec y Tianguismanalco;
  - 2021: Ángel Albino Corzo, Emiliano Zapata, Nextlalpan y Coxquihui;
  - 2024: Ocampo, Olintla, San Lucas Tecopilco y Tlilapan;
  - Ixil (Yucatán) en 2015.
- **Cuando el paquete se perdió para más de la mitad de la lista,** el año queda "sin dato", con una marca de conflicto.

**Interpretación para el reto:** los de origen A tienen un problema distinto al de los demás. No es que la gente no quiera votar: **no tuvo dónde votar**, por decisión comunitaria, conflicto o violencia, según el caso. Una campaña para motivar el voto no les sirve; necesitan otro tipo de atención del INE.

**Cuidado:** eso no los vuelve automáticamente de riesgo máximo. Varios votan bien cuando sí hay casillas: Solosuchiapa tuvo 79 % en 2021 y 2024, y Bolaños, 76 % en 2024. Por eso proponemos una **alerta propia** basada en sus antecedentes, separada del modelo (sección 8, D12).

## 4 · Lo que el modelo va a anticipar: la desviación de cada municipio

El modelo separa dos preguntas:

1. **¿Cuánto votará el país en 2027?** Es un **escenario**, no algo que se pueda predecir con 6 elecciones. Usamos como referencia las intermedias anteriores (44.1 %, 47.1 % y 51.8 %).
2. **¿Qué municipios quedarán por debajo del resto?** Eso sí se puede anticipar con la historia de cada municipio. Lo medimos con la **desviación**: participación del municipio menos la nacional, en puntos.

Así, el riesgo de cada municipio en 2027 es: **escenario nacional + desviación esperada del municipio**.

In [13]:
nacional = panel.group_by('AELEC').agg(
    participacion_nacional=pl.col('SV').sum() / (pl.col('SV').sum() + pl.col('NV').sum()) * 100
).sort('AELEC')
panel = panel.join(nacional, on='AELEC').with_columns(
    tipo_eleccion=pl.col('AELEC').replace_strict(TIPO),
    desviacion=pl.col('participacion') - pl.col('participacion_nacional'),
)
nacional.with_columns(pl.col('participacion_nacional').round(2)).rename({'AELEC': 'Elección', 'participacion_nacional': 'Participación nacional (%)'})

Elección,Participación nacional (%)
i16,f64
2009,44.09
2012,62.1
2015,47.06
2018,62.38
2021,51.85
2024,59.68


**Qué vemos:** las cifras nacionales son las del paso 1 (D1), con diferencias de 0.03 pp o menos (por ejemplo, 2009: 44.09 % contra 44.06 %). Vienen de las secciones que no cuadran con las actas (D10 ajustada).

### ¿Qué tanto se repite la posición de cada municipio?

Si un municipio quedó por debajo del país en una elección, ¿vuelve a quedar por debajo en la siguiente? Medimos el R² entre las desviaciones de dos elecciones. Lo calculamos dos veces:

- con todos los municipios;
- quitando a los que, **antes** de la elección a anticipar, ya habían tenido parte de su lista sin casilla. Es lo único que el INE sabría de antemano: nadie sabe hoy dónde faltarán casillas en 2027.

In [14]:
panel = panel.with_columns(sin_casilla=pl.col('origen').str.starts_with('A').fill_null(False))
anios_sin_casilla = panel.filter(pl.col('sin_casilla')).select('cvegeo', 'AELEC')
ancho = panel.pivot(on='AELEC', index='cvegeo', values='desviacion')

def r2(tabla, objetivo, pista):
    par = tabla.select(objetivo, pista).drop_nulls()
    return float(np.corrcoef(par[objetivo], par[pista])[0, 1] ** 2), par.height

filas = []
for objetivo, pista in [('2015', '2012'), ('2015', '2009'), ('2021', '2018'), ('2021', '2015'), ('2024', '2021'), ('2024', '2018')]:
    todos, n = r2(ancho, objetivo, pista)
    # Solo lo que se sabía antes de la elección a anticipar: municipios con lista sin casilla en una elección anterior
    con_antecedente = anios_sin_casilla.filter(pl.col('AELEC') < int(objetivo))['cvegeo'].unique().to_list()
    sin_antecedente, _ = r2(ancho.filter(~pl.col('cvegeo').is_in(con_antecedente)), objetivo, pista)
    filas.append({'Elección a anticipar': f'{objetivo} ({TIPO[int(objetivo)]})', 'Con la elección de': f'{pista} ({TIPO[int(pista)]})',
                  'R² (todos)': round(todos, 2), 'R² (sin municipios con antecedente de lista sin casilla)': round(sin_antecedente, 2),
                  'Municipios con antecedente': len(con_antecedente), 'Municipios': n})
pl.DataFrame(filas)

Elección a anticipar,Con la elección de,R² (todos),R² (sin municipios con antecedente de lista sin casilla),Municipios con antecedente,Municipios
str,str,f64,f64,i64,i64
"""2015 (intermedia)""","""2012 (presidencial)""",0.4,0.41,9,2455
"""2015 (intermedia)""","""2009 (intermedia)""",0.4,0.41,9,2462
"""2021 (intermedia)""","""2018 (presidencial)""",0.66,0.68,54,2462
"""2021 (intermedia)""","""2015 (intermedia)""",0.26,0.3,54,2455
"""2024 (presidencial)""","""2021 (intermedia)""",0.63,0.64,68,2468
"""2024 (presidencial)""","""2018 (presidencial)""",0.67,0.69,68,2467


**Qué vemos:**

- **Para 2021, la presidencial anterior fue la mejor pista:** 2018 anticipa dos tercios de las diferencias entre municipios (R² 0.66), y 2015, apenas una cuarta parte (0.26).
- **Para 2015 no hay una pista mejor que otra:** 2012 y 2009 empatan (0.40). Es la intermedia del boicot magisterial: ahora que sus municipios tienen la participación de las actas (muy baja), 2015 es todavía más difícil de anticipar que en el paso 1.
- **Quitar a los municipios con antecedentes cambia poco** (en 2021, de 0.66 a 0.68). La mayoría de las faltas de casilla no se veían venir en la historia de esos municipios.
- **Con todo, alrededor de un tercio de las diferencias entre municipios queda sin explicar** con la sola historia. Las siguientes secciones buscan qué más ayuda.

**Para 2027:** la base es 2024 (presidencial). Si agregar 2021 (intermedia) mejora la predicción, se probará en la validación del paso 4.

## 5 · Elecciones locales el mismo día

En el paso 2 vimos que **elegir gobernador el mismo día** cambia mucho la participación del estado en la intermedia. Ahora tenemos el calendario completo, con fuentes, en `data/reference/elecciones_locales_concurrentes.csv`. Indica, para cada estado y elección (2009 a 2027), si el mismo día se eligió **gubernatura**, **diputaciones locales** y **ayuntamientos**. Las fuentes y los casos especiales están en `docs/investigacion/I-elecciones-concurrentes.md`.

**Excepciones municipales documentadas:**

- **Oaxaca:** 418 municipios eligen a sus autoridades por sistemas normativos indígenas (usos y costumbres), en asambleas y fechas propias. Por eso no eligen ayuntamiento el mismo día. La lista viene del catálogo oficial del IEEPCO (`data/reference/oaxaca_municipios_sni.csv`). San Baltazar Chichicápam entró a ese régimen en 2024, así que lo tratamos como municipio de partidos hasta 2024.
- **Hidalgo 2021:** no hubo elección ordinaria de ayuntamientos, pero Acaxochitlán e Ixmiquilpan tuvieron elección extraordinaria ese mismo día.
- **Chiapas 2024:** Chicomuselo, Pantelhó y Capitán Luis Ángel Vidal eligieron ayuntamiento en una extraordinaria el 25 de agosto, no el día de la elección federal.

In [15]:
concurrencia_estatal = pl.read_csv(raiz / 'data/reference/elecciones_locales_concurrentes.csv', infer_schema=False).select(
    pl.col('anio').cast(pl.Int16).alias('AELEC'),
    pl.col('cve_ent').cast(pl.Int8).alias('clave_estado'),
    (pl.col('gubernatura') == 'si').alias('gobernador_mismo_dia'),
    (pl.col('diputaciones_locales') == 'si').alias('diputacion_local_mismo_dia'),
    (pl.col('ayuntamientos') == 'si').alias('ayuntamiento_mismo_dia'),
)
municipios_sni = pl.read_csv(raiz / 'data/reference/oaxaca_municipios_sni.csv', infer_schema=False)['cvegeo_inegi'].to_list()
assert len(municipios_sni) == 418

def clave(estado, nombre):
    encontrado = nombres_2024.filter(pl.col('cvegeo').str.starts_with(estado) & (pl.col('municipio') == nombre))['cvegeo'].to_list()
    assert len(encontrado) == 1, (estado, nombre, encontrado)
    return encontrado[0]

extraordinaria_mismo_dia = {2021: [clave('13', 'Acaxochitlán'), clave('13', 'Ixmiquilpan')]}
extraordinaria_otro_dia = {2024: [clave('07', 'Chicomuselo'), clave('07', 'Pantelhó'), clave('07', 'Capitán Luis Ángel Vidal')]}
chichicapam = clave('20', 'San Baltazar Chichicápam')

ANIOS_CON_2027 = ANIOS + [2027]
concurrencia = (
    nombres_2024.select('cvegeo').join(pl.DataFrame({'AELEC': ANIOS_CON_2027}, schema={'AELEC': pl.Int16}), how='cross')
    .with_columns(clave_estado=pl.col('cvegeo').str.slice(0, 2).cast(pl.Int8))
    .join(concurrencia_estatal, on=['AELEC', 'clave_estado'], how='left', validate='m:1')
    .with_columns(
        es_sni=pl.col('cvegeo').is_in(municipios_sni) & ~((pl.col('cvegeo') == chichicapam) & (pl.col('AELEC') <= 2024)),
    )
    .with_columns(
        ayuntamiento_mismo_dia=pl.when(pl.col('es_sni')).then(False)
        .when(pl.struct('AELEC', 'cvegeo').map_elements(lambda r: r['cvegeo'] in extraordinaria_mismo_dia.get(r['AELEC'], []), return_dtype=pl.Boolean)).then(True)
        .when(pl.struct('AELEC', 'cvegeo').map_elements(lambda r: r['cvegeo'] in extraordinaria_otro_dia.get(r['AELEC'], []), return_dtype=pl.Boolean)).then(False)
        .otherwise(pl.col('ayuntamiento_mismo_dia')),
    )
)
assert concurrencia['gobernador_mismo_dia'].null_count() == 0
panel = panel.join(concurrencia.drop('clave_estado'), on=['cvegeo', 'AELEC'], how='left')
panel = panel.with_columns(clave_estado=pl.col('cvegeo').str.slice(0, 2).cast(pl.Int8))

resumen = concurrencia_estatal.group_by('AELEC').agg(
    pl.col('gobernador_mismo_dia').sum().alias('Estados con gubernatura'),
    pl.col('diputacion_local_mismo_dia').sum().alias('Estados con diputaciones locales'),
    pl.col('ayuntamiento_mismo_dia').sum().alias('Estados con ayuntamientos'),
).join(
    concurrencia.group_by('AELEC').agg((pl.col('ayuntamiento_mismo_dia').mean() * 100).round(0).alias('% de municipios que eligen ayuntamiento')), on='AELEC'
).sort('AELEC').with_columns(pl.col('AELEC').replace_strict({**TIPO, 2027: 'intermedia'}).alias('Tipo')).rename({'AELEC': 'Elección'})
resumen

Elección,Estados con gubernatura,Estados con diputaciones locales,Estados con ayuntamientos,% de municipios que eligen ayuntamiento,Tipo
i16,u32,u32,u32,f64,str
2009,6,11,11,23.0,"""intermedia"""
2012,7,15,15,36.0,"""presidencial"""
2015,9,16,16,36.0,"""intermedia"""
2018,9,27,25,66.0,"""presidencial"""
2021,15,30,30,78.0,"""intermedia"""
2024,9,31,30,73.0,"""presidencial"""
2027,17,31,30,73.0,"""intermedia"""


**Qué vemos:**

- **Desde la reforma de 2014, casi todo el país elige algo local el mismo día que la federal.** En 2009, 2012 y 2015, la mitad de los estados o menos.
- **2027 se parece mucho a 2021:** 17 gubernaturas (15 en 2021), diputaciones locales en 31 estados y ayuntamientos en 30.
- **2015 es muy distinta:** solo el 36 % de los municipios eligió ayuntamiento el mismo día.

### ¿Cuánto cambia la desviación de cada municipio según lo que se elige?

Clasificamos cada municipio, en cada elección, en tres niveles:

- **con gobernador:** se eligió gobernador (y casi siempre también congreso o ayuntamientos);
- **locales sin gobernador:** se eligieron diputaciones locales o ayuntamientos, pero no gobernador;
- **ninguna:** no hubo elección local ese día.

Luego medimos cuánto cambió la desviación de cada municipio de la presidencial a la intermedia siguiente, según cómo cambió su nivel. Dejamos fuera los años con parte de la lista sin casilla (origen A). Como Oaxaca tiene 570 municipios y pesa mucho en las medianas, mostramos también el resultado sin Oaxaca.

In [16]:
nivel_local = (
    pl.when(pl.col('gobernador_mismo_dia')).then(pl.lit('con gobernador'))
    .when(pl.col('ayuntamiento_mismo_dia') | pl.col('diputacion_local_mismo_dia')).then(pl.lit('locales sin gobernador'))
    .otherwise(pl.lit('ninguna'))
)
concurrencia = concurrencia.with_columns(nivel_eleccion_local=nivel_local)
panel = panel.join(concurrencia.select('cvegeo', 'AELEC', 'nivel_eleccion_local'), on=['cvegeo', 'AELEC'], how='left')

def cambio_por_nivel(presidencial, intermedia):
    antes = panel.filter(pl.col('AELEC') == presidencial).select('cvegeo', 'clave_estado', pl.col('desviacion').alias('antes'), pl.col('nivel_eleccion_local').alias('En la presidencial'))
    despues = panel.filter(pl.col('AELEC') == intermedia).select('cvegeo', pl.col('desviacion').alias('despues'), pl.col('nivel_eleccion_local').alias('En la intermedia'))
    return (
        antes.join(despues, on='cvegeo')
        .join(anios_sin_casilla.filter(pl.col('AELEC').is_in([presidencial, intermedia])), on='cvegeo', how='anti')
        .drop_nulls(['antes', 'despues'])
        .group_by('En la presidencial', 'En la intermedia').agg(
            pl.col('clave_estado').n_unique().alias('Estados'),
            pl.len().alias('Municipios'),
            (pl.col('despues') - pl.col('antes')).median().round(1).alias('Cambio típico de la desviación (pp)'),
            (pl.col('despues') - pl.col('antes')).filter(pl.col('clave_estado') != 20).median().round(1).alias('Ídem, sin Oaxaca (pp)'),
        ).with_columns(pl.lit(f'{presidencial} → {intermedia}').alias('Periodo'))
    )

pl.concat([cambio_por_nivel(2012, 2015), cambio_por_nivel(2018, 2021)]).select(
    'Periodo', 'En la presidencial', 'En la intermedia', 'Estados', 'Municipios', 'Cambio típico de la desviación (pp)', 'Ídem, sin Oaxaca (pp)'
).sort('Periodo', 'Cambio típico de la desviación (pp)', descending=[False, True])

Periodo,En la presidencial,En la intermedia,Estados,Municipios,Cambio típico de la desviación (pp),"Ídem, sin Oaxaca (pp)"
str,str,str,u32,u32,f64,f64
"""2012 → 2015""","""ninguna""","""con gobernador""",2,115,27.0,27.0
"""2012 → 2015""","""locales sin gobernador""","""con gobernador""",7,306,14.5,14.5
"""2012 → 2015""","""con gobernador""","""locales sin gobernador""",6,345,12.9,12.9
"""2012 → 2015""","""locales sin gobernador""","""locales sin gobernador""",1,125,6.4,6.4
"""2012 → 2015""","""ninguna""","""ninguna""",15,1404,-1.7,-2.9
"""2012 → 2015""","""con gobernador""","""ninguna""",1,122,-2.7,-2.7
"""2018 → 2021""","""ninguna""","""con gobernador""",2,26,12.5,12.5
"""2018 → 2021""","""con gobernador""","""locales sin gobernador""",9,877,7.3,7.3
"""2018 → 2021""","""locales sin gobernador""","""con gobernador""",13,621,6.7,6.7


**Qué vemos:**

- **De 2012 a 2015, lo que más pesó fue tener o no tener elección local en la intermedia.**
  - Los municipios que pasaron de no tener nada a elegir gobernador (Baja California Sur y Michoacán) subieron 27 pp frente al país.
  - Los que no tuvieron elección local en ninguno de los dos años (15 estados) quedaron igual o un poco peor.
  - Chiapas eligió gobernador en 2012 y votó lo local en otra fecha en 2015: cayó 2.7 pp.
- **De 2018 a 2021, con casi todo el país eligiendo algo local, el gobernador pesó menos.**
  - Los que estrenaron gobernador en 2021 subieron 6.7 pp.
  - Los que habían elegido gobernador en 2018 subieron 7.3 pp.
  - **Los que tuvieron elecciones locales sin gobernador en los dos años (8 estados) casi no se movieron:** +0.8 pp, o +1.9 pp sin Oaxaca. El rezago se mantiene sin Oaxaca.

**Por eso corregimos lo que dijimos en el paso 2:** el contraste de −4.3 contra −17.6 pp entre 2012 y 2015 no era solo efecto del gobernador. Era sobre todo tener o no tener **alguna** elección local ese día.

**Cuidado:** por qué suben tanto los que **dejaron** de elegir gobernador en 2021 no lo sabemos. Lo reportamos como dato, sin explicarlo.

### 2027 se parece a 2021

¿En qué grupo cae cada municipio en la transición de 2024 a 2027?

In [17]:
transicion_2027 = (
    concurrencia.filter(pl.col('AELEC') == 2024).select('cvegeo', pl.col('nivel_eleccion_local').alias('En 2024'))
    .join(concurrencia.filter(pl.col('AELEC') == 2027).select('cvegeo', 'clave_estado', pl.col('nivel_eleccion_local').alias('En 2027')), on='cvegeo')
    .join(nombres_2024.select('cvegeo', 'estado'), on='cvegeo')
)
transicion_2027.group_by('En 2024', 'En 2027').agg(
    pl.col('clave_estado').n_unique().alias('Estados'), pl.len().alias('Municipios'), pl.col('estado').unique().sort().str.join(', ').alias('Cuáles')
).sort('En 2024', 'En 2027')

En 2024,En 2027,Estados,Municipios,Cuáles
str,str,u32,u32,str
"""con gobernador""","""locales sin gobernador""",9,899,"""Chiapas, Ciudad de México, Guanajuato, Jalisco, Morelos, Puebla, Tabasco, Veracruz de Igna…"
"""locales sin gobernador""","""con gobernador""",17,679,"""Aguascalientes, Baja California, Baja California Sur, Campeche, Chihuahua, Colima, Guerrer…"
"""locales sin gobernador""","""locales sin gobernador""",6,899,"""Coahuila de Zaragoza, Durango, Hidalgo, México, Oaxaca, Tamaulipas"""


**Qué vemos:**

- **La transición de 2024 a 2027 tiene los mismos tres grupos principales que la de 2018 a 2021.** No hay ningún municipio que pase de "ninguna" a tener elección local, como ocurrió en 2015 y, en menor medida, en 2021 (27 municipios de Baja California y Nayarit).
- **El grupo que casi no se movió en 2021** (elecciones locales sin gobernador en los dos años) **en 2027 lo forman Coahuila, Durango, Hidalgo, Estado de México, Oaxaca y Tamaulipas.** Los seis ya estaban en ese grupo en 2021, junto con Aguascalientes y Quintana Roo, que en 2027 sí eligen gobernador.

**Interpretación para el reto:**

- **Para 2027, la transición de 2018 a 2021 es el mejor ejemplo del pasado.** Por eso el modelo debe aprender sobre todo de esa transición, y probarse también con 2015 para medir su incertidumbre (D15).
- **Hipótesis para el modelo, no conclusión:** los municipios de esos seis estados podrían quedar por debajo del resto en 2027, porque no tendrán el empujón de una gubernatura. Hay que confirmarlo en el paso 4.

**Cuidados:**

- **Oaxaca pesa mucho en ese grupo:** son 570 de sus 899 municipios, y la mayoría elige por usos y costumbres.
- **Las medianas son por municipio, no por persona:** un municipio pequeño pesa lo mismo que uno grande.
- **Es una comparación, no un experimento:** los estados que eligen gobernador pueden diferir en otras cosas.

## 6 · ¿Quién está en la lista? Rural, jóvenes, hombres y crecimiento

Medimos si cuatro características de la lista de cada municipio en la elección anterior (2018) se relacionan con su desviación en la intermedia de 2021:

| Variable | Cómo se calcula |
|---|---|
| **% rural** | Lista en secciones rurales / lista total |
| **% jóvenes** | Lista de 20 a 29 años / lista total |
| **% hombres** | Lista de hombres / lista total |
| **Crecimiento de la lista** | Cambio de la lista de 2018 a 2021, en % (variable continua, D11) |

La medida es la **correlación** (de −1 a 1; 0 = ninguna relación). También medimos si ayudan a explicar **lo que la historia no explica** (la parte de 2021 que 2018 no anticipa).

In [18]:
composicion = panel.filter(pl.col('AELEC') == 2018).select(
    'cvegeo',
    pct_rural=pl.col('LN_rural') / pl.col('LN') * 100,
    pct_jovenes=pl.col('LN_20_29') / pl.col('LN') * 100,
    pct_hombres=pl.col('LN_hombres') / pl.col('LN') * 100,
)
crecimiento = panel.pivot(on='AELEC', index='cvegeo', values='LN').select('cvegeo', crecimiento_lista=(pl.col('2021') / pl.col('2018') - 1) * 100)
t = (
    ancho.join(anios_sin_casilla.filter(pl.col('AELEC').is_in([2018, 2021])), on='cvegeo', how='anti')
    .join(composicion, on='cvegeo').join(crecimiento, on='cvegeo')
    .drop_nulls(['2021', '2018'])
)
# Lo que 2018 no anticipa de 2021 (residuo de una recta)
pendiente, ordenada = np.polyfit(t['2018'], t['2021'], 1)
t = t.with_columns(no_explicado=pl.col('2021') - (ordenada + pendiente * pl.col('2018')))

def correlacion(a, b):
    par = t.select(a, b).drop_nulls().filter(pl.all_horizontal(pl.all().is_finite()))
    return round(float(np.corrcoef(par[a], par[b])[0, 1]), 2)

etiquetas = {'pct_rural': '% rural', 'pct_jovenes': '% jóvenes (20 a 29)', 'pct_hombres': '% hombres', 'crecimiento_lista': 'Crecimiento de la lista 2018 → 2021'}
pl.DataFrame([
    {'Variable': nombre, 'Correlación con la desviación de 2021': correlacion('2021', v),
     'Correlación con lo que 2018 no anticipa': correlacion('no_explicado', v)}
    for v, nombre in etiquetas.items()
])

Variable,Correlación con la desviación de 2021,Correlación con lo que 2018 no anticipa
str,f64,f64
"""% rural""",0.18,0.14
"""% jóvenes (20 a 29)""",0.02,-0.05
"""% hombres""",0.27,0.24
"""Crecimiento de la lista 2018 → 2021""",0.02,-0.04


**Qué vemos:**

- **Ninguna de las cuatro variables tiene una relación fuerte** con la desviación de 2021. La más alta es la de % hombres (0.27), y aun así es débil.
- **% jóvenes y crecimiento de la lista: prácticamente cero.** Que un municipio tenga más jóvenes no lo hace votar menos que el resto, aunque los jóvenes sean el grupo que menos vota.
- **% hombres (0.24) y % rural (0.14) aportan algo a lo que la historia no anticipa.**

**Hipótesis, no verificada, sobre % hombres:** donde emigran muchos hombres, la lista tiene menos hombres y sigue contando a personas que viven fuera y no pueden votar en su sección, lo que baja la participación. Habría que comprobarlo con datos de migración, por ejemplo el índice de intensidad migratoria del CONAPO.

**Interpretación:** casi todo lo que estas variables dicen ya está "dentro" de la historia. Si un municipio rural vota más, eso ya se ve en su desviación de 2018. Por eso **no conviene llenar el modelo de variables**: cada una debe ganarse su lugar mejorando la predicción en la validación (paso 4).

**Por qué igual importan los jóvenes:** no para anticipar **dónde** está el riesgo, sino para diseñar la **intervención**. El escenario "con campaña dirigida" necesita saber cuántos jóvenes hay en cada municipio y cuánto votan. Esa información ya queda en la tabla (sección 7).

## 7 · Los jóvenes en cada municipio (para el escenario de intervención)

Para el escenario "con campaña dirigida" necesitamos, por municipio, cuántos jóvenes hay y cuánto votaron. Lo calculamos para la última intermedia (2021).

In [19]:
jovenes_2021 = panel.filter((pl.col('AELEC') == 2021) & pl.col('participacion').is_not_null() & (pl.col('fuente_participacion') == 'conteos censales')).with_columns(
    participacion_20_29=pl.col('SV_20_29') / (pl.col('SV_20_29') + pl.col('NV_20_29')) * 100,
    participacion_h_20_29=pl.col('SV_h_20_29') / (pl.col('SV_h_20_29') + pl.col('NV_h_20_29')) * 100,
).with_columns(brecha_jovenes=pl.col('participacion_20_29') - pl.col('participacion'))
print('Municipios:', jovenes_2021.height)
jovenes_2021.select(
    pl.col('participacion').median().round(1).alias('Participación total (mediana, %)'),
    pl.col('participacion_20_29').median().round(1).alias('Jóvenes 20 a 29 (mediana, %)'),
    pl.col('participacion_h_20_29').median().round(1).alias('Hombres 20 a 29 (mediana, %)'),
    pl.col('brecha_jovenes').median().round(1).alias('Brecha jóvenes − total (mediana, pp)'),
    (pl.col('brecha_jovenes') < 0).mean().mul(100).round(1).alias('% de municipios donde los jóvenes votan menos'),
)

Municipios: 2427


"Participación total (mediana, %)","Jóvenes 20 a 29 (mediana, %)","Hombres 20 a 29 (mediana, %)","Brecha jóvenes − total (mediana, pp)",% de municipios donde los jóvenes votan menos
f64,f64,f64,f64,f64
61.3,50.9,43.1,-9.9,97.5


**Qué vemos:**

- **En el municipio típico, las personas de 20 a 29 años votaron 50.9 % en 2021**, contra 61.3 % del total: una brecha de 9.9 pp.
- **Los hombres de 20 a 29 votaron todavía menos:** 43.1 %.
- **Los jóvenes votan menos que el total en el 97.5 % de los municipios.**

(Medianas de los municipios cuya participación sale de los conteos censales. Usamos el grupo de 20 a 29 años que destaca la decisión D8.)

**Interpretación:** el grupo a intervenir es el mismo en casi todo el país. Lo que cambia entre municipios es **cuántos jóvenes hay** y **qué tan grande es su brecha**. Con eso se puede estimar, en el paso 5, cuánto bajaría la abstención si una campaña cerrara una parte de esa brecha.

## 8 · Decisiones

**Decididas por Sofía Ordaz el 19 de octubre de 2026:** se adoptan todas las recomendaciones de esta tabla.

| # | Decisión | Opciones | Recomendación adoptada y por qué |
|---|---|---|---|
| **D2 (ajustada)** | Regla 3 de "lista incompleta" | (a) como está; (b) solo cuando hay elección antes y después, con la media geométrica de las dos | **(b).** En 2009 solo hay una elección para comparar, y con la media simple se marcaba como incompletos a municipios que crecieron de verdad (García, Juárez, Solidaridad, Pesquería). En 2012, que es donde importa, sigue marcando 108 municipios |
| **D3 (ajustada otra vez)** | Cuándo se usa la participación de las actas | (a) solo con más de la mitad en "no sabemos"; (b) además, siempre que 10 % o más de la lista se quede sin casilla | **(b).** Con (a), municipios donde parte de la gente no tuvo dónde votar aparecían votando más que el país (sección 3) |
| **D9 (ampliada)** | Actas como segunda fuente | Se agregan las actas oficiales de 2009, 2012 y 2015 (SICEEF del INE) | Aplicación directa de D9 a datos nuevos: permite clasificar el boicot de 2015 y los cuadernillos no capturados de 2012 |
| **D16 (nueva, propuesta)** | Municipios con lista incompleta en los conteos | (a) dejarlos "sin dato"; (b) usar la participación de todas sus actas | **(b).** Recupera 108 municipios de 2012. **Pendiente de confirmar por Sofía** |
| **D10 (ajustada)** | Secciones que no cuadran con el acta | (a) solo cuando marcan de más; (b) en los dos sentidos | **(b).** Marcar de menos fabrica abstención que no existió, que es justo lo que el reto busca |
| **D12** | Municipios con parte de su lista sin casilla | (a) tratarlos como a los demás; (b) **alerta propia**, calculada solo con elecciones anteriores: "recurrente" si pasó en 2 o más elecciones, "antecedente" si pasó en una. En el entrenamiento del modelo se excluye solo el año sin casilla, no el municipio completo | **(b).** Su problema es distinto (no hubo dónde votar) y una campaña no lo resuelve. No los ponemos en riesgo máximo automáticamente: varios votan bien cuando hay casillas |
| **D13 (ajustada)** | "Territorio aproximado" | (a) marcar a todo municipio que cedió territorio; (b) marcar solo cuando el que recibió no tiene todo su territorio ese año, y marcar a los dos | **(b).** Si el que recibió tiene el 100 % de su territorio, la historia de los dos es exacta. Quedan 13 municipios con duda real, sobre todo Apodaca–Guadalupe y Puente de Ixtla–Xoxocotla. Se conservan con la marca |
| **D14** | Variables del modelo | (a) meter todas; (b) empezar con la historia más el nivel de elección local el mismo día (en la presidencial y en la intermedia), y agregar otras (% hombres, % rural) solo si mejoran la validación | **(b).** Las variables de la lista casi no aportan por sí solas. Un modelo simple es más fácil de explicar al INE y más difícil de sobreajustar |
| **D15** | Cómo validar | (a) entrenar con la transición 2012 → 2015 y probar en 2018 → 2021, respetando el orden del tiempo; (b) mezclar años al azar | **(a).** Así se simula lo que haremos en 2027: predecir una intermedia que el modelo no ha visto. Vamos a medir: <br>• el error en pp; <br>• cuántos coinciden en el 10 % de mayor riesgo; <br>• el resultado ponderado por la lista, porque al INE le importan las personas, no solo los municipios. <br>Lo comparamos contra una regla simple justa: la desviación de la presidencial anterior más el cambio típico de la presidencial a la intermedia. Como solo hay dos intermedias para aprender, reportamos la incertidumbre |

## 9 · La tabla para el modelo

Guardamos la tabla municipio × elección en `data/processed/panel_municipal.parquet`. No se sube al repositorio (`data/*` está en `.gitignore`): este notebook la regenera.

In [20]:
columnas = {
    'cvegeo': 'Clave INEGI del municipio (territorio de 2024)',
    'estado': 'Estado',
    'municipio': 'Municipio (nombre oficial del INEGI)',
    'AELEC': 'Año de la elección',
    'tipo_eleccion': 'presidencial o intermedia',
    'LN': 'Lista nominal',
    'SV': 'Votó (conteos censales, con D10 aplicado)',
    'NV': 'No votó',
    'NS': 'No sabemos',
    'pct_no_sabemos': '% de la lista en "no sabemos"',
    'participacion_conteo': 'Participación con la fórmula de los conteos censales (D1)',
    'origen': 'A: 10 % o más de la lista sin casilla; B: paquete no recibido; C: cuadernillo no capturado (solo 2018, 2021 y 2024)',
    'sin_casilla': 'Ese año, 10 % o más de la lista se quedó sin casilla (origen A)',
    'pct_sin_casilla': '% de la lista que no cubrió ninguna casilla que funcionara, medido en el municipio completo (actas)',
    'pct_paquete': '% de la lista con paquete no recibido (actas)',
    'participacion_actas': 'Participación según actas: votos / (lista − lista con paquete perdido)',
    'motivo_sin_dato': 'Por qué el año queda sin dato (D2)',
    'participacion': 'Participación final (la que usa el modelo)',
    'fuente_participacion': 'De dónde sale la participación final',
    'participacion_nacional': 'Participación nacional del año',
    'desviacion': 'Participación final − nacional (pp)',
    'territorio_aproximado': 'Parte de su lista de ese año no se puede ubicar con certeza entre él y otro municipio (D13 ajustada)',
    'secciones_que_no_cuadran': 'Secciones que no cuadran con el acta, en cualquier sentido (D10 ajustada)',
    'secciones_sin_casilla': 'Secciones sin ninguna casilla en las actas',
    'LN_secciones_desaparecidas': 'Lista en secciones que ya no existen en 2024',
    'LN_rural': 'Lista en secciones rurales',
    'LN_20_29': 'Lista de 20 a 29 años', 'SV_20_29': 'Votó, 20 a 29', 'NV_20_29': 'No votó, 20 a 29',
    'LN_h_20_29': 'Lista de hombres de 20 a 29', 'SV_h_20_29': 'Votó, hombres 20 a 29', 'NV_h_20_29': 'No votó, hombres 20 a 29',
    'LN_65_mas': 'Lista de 65 años o más', 'LN_hombres': 'Lista de hombres',
    'clave_estado': 'Clave INEGI del estado',
    'gobernador_mismo_dia': 'Se eligió gobernador el mismo día',
    'diputacion_local_mismo_dia': 'Se eligieron diputaciones locales el mismo día',
    'ayuntamiento_mismo_dia': 'El municipio eligió ayuntamiento el mismo día (con excepciones SNI y extraordinarias)',
    'es_sni': 'Municipio de Oaxaca que elige por sistemas normativos indígenas',
    'nivel_eleccion_local': 'Qué se eligió localmente ese día: con gobernador, locales sin gobernador o ninguna',
}
salida = raiz / 'data/processed/panel_municipal.parquet'
panel.select(list(columnas)).write_parquet(salida)
print(f'Guardado: {salida.relative_to(raiz)} · {panel.height:,} filas · {len(columnas)} columnas')
# Calendario municipal 2009-2027 (incluye 2027, que el modelo necesita)
concurrencia.write_parquet(raiz / 'data/processed/concurrencia_municipal.parquet')
# Tabla por sección: a qué municipio de 2024 va y si cuadra con su acta (para cruces por edad y sexo)
secciones.select('AELEC', 'EDOCVE', 'MPIOCVE', 'SECCION', 'TIPOSEC', 'municipio_2024', 'seccion_existe_en_2024', 'hay_actas', 'actas_casillas', 'no_cuadra_con_acta').write_parquet(raiz / 'data/processed/secciones_territorio_2024.parquet')
pl.DataFrame({'Columna': list(columnas), 'Qué contiene': list(columnas.values())})

Guardado: data/processed/panel_municipal.parquet · 14,862 filas · 40 columnas


Columna,Qué contiene
str,str
"""cvegeo""","""Clave INEGI del municipio (territorio de 2024)"""
"""estado""","""Estado"""
"""municipio""","""Municipio (nombre oficial del INEGI)"""
"""AELEC""","""Año de la elección"""
"""tipo_eleccion""","""presidencial o intermedia"""
"""LN""","""Lista nominal"""
"""SV""","""Votó (conteos censales, con D10 aplicado)"""
"""NV""","""No votó"""
"""NS""","""No sabemos"""


## Siguiente paso

**Paso 4 · Modelo y validación (primer resultado el 22 de octubre):**

1. Con las decisiones de la sección 8 (ya tomadas), entrenar el modelo de desviación con la transición 2012 → 2015 y probarlo en 2018 → 2021 (D15).
2. Compararlo con la regla simple justa y medir la incertidumbre.
3. Producir la primera lista de municipios en riesgo para 2027, con la alerta aparte para los que tienen antecedentes de lista sin casilla (D12).